### seeds → RAW → STAGING → MARTS → FACT/DIM

Initialize dbt project 

In [ ]:
dbt init project

project/
 ├── models/
 ├── macros/
 ├── seeds/
 ├── tests/
 ├── snapshots/
 ├── dbt_project.yml

In [ ]:
pip install dbt-snowflake

Configure Snowflake connection

In [ ]:
C:\Users\ACER\.dbt\profiles.yml

project:
  target: dev
  outputs:
    dev:
      type: snowflake
      account: yvqzwkk-itb60254
      user: dbt
      password: dbtPassword123
      role: TRANSFORM
      database: DBT_BI_METRICS_PLATFORM_DB
      warehouse: COMPUTE_WH
      schema: RAW
      threads: 1

In [ ]:
dbt debug

dbt_project.yml configuration

In [ ]:
project/dbt_project.yml

name: 'project'
version: '1.0.0'
profile: 'project'

model-paths: ["models"]
seed-paths: ["seeds"]
macro-paths: ["macros"]

models:
  project:
    staging:
      +schema: RAW
    marts:
      +schema: MARTS

Schema result: \
RAW_MARTS

Because dbt uses: \
target.schema + "_" + custom_schema

Load dataset using seeds

In [ ]:
seeds/sales.csv

dbt seed

RAW.sales

Create staging model

In [ ]:
models/staging/stg_sales.

{{ config(materialized='table') }}

select *
from {{ ref('sales') }}

Create marts

In [ ]:
sales_summary.sql

{{ config(materialized='table') }}

select
    city,
    sum(total) as total_sales
from {{ ref('stg_sales') }}
group by city

Macros in dbt

In [ ]:
macros/agg.sql

{% macro agg_metrics() %}

sum(total) as total_sales,
sum(quantity) as total_qty

{% endmacro %}

use-
select
    city,
    {{ agg_metrics() }}
from {{ ref('stg_sales') }}

Incremental model

In [ ]:
models/marts/fact_sales.sql

{{ config(
    materialized='incremental',
    unique_key='invoice_id'
) }}

select *
from {{ ref('stg_sales') }}

{% if is_incremental() %}

where date > (select max(date) from {{ this }})

{% endif %}

Audit macro

In [ ]:
macros/audit.sql

{% macro audit_cols() %}

current_timestamp as created_at,
current_timestamp as updated_at

{% endmacro %}

use-
select
    *,
    {{ audit_cols() }}
from {{ ref('stg_sales') }}

Fact table macro

In [ ]:
macros/fact_cols.sql

{% macro fact_cols() %}

invoice_id,
branch,
city,
total,
quantity,
date

{% endmacro %}

for model:
fact_sales.sql

select
    {{ fact_cols() }},
    {{ audit_cols() }}
from {{ ref('stg_sales') }}

How dbt works internally

In [ ]:
dbt run

Parse models

Parse macros

Compile SQL

Resolve ref()

Resolve macros

Generate SQL

Send to Snowflake

Create tables

Compiled SQL stored in:target/compiled/

Commands used in project

In [ ]:
dbt init
dbt debug
dbt seed
dbt run
dbt test
dbt compile
dbt clean